# 🧠 SLD-VM: Full Pipeline — Hard Reasoning Benchmarks
**Structured Latent Deliberation with Verifiable Memory**

## Datasets
| Dataset | Task | Difficulty | Why hard |
|---------|------|-----------|----------|
| **MATH** (Hendrycks) | Competition Math | ⭐⭐⭐⭐⭐ | Multi-step algebra/number theory, ~35-50% small model baseline |
| **ARC-Challenge** | Science MCQ | ⭐⭐⭐⭐ | Hard science reasoning, ~45-65% small model baseline, random=25% |

## Pipeline Stages
```
Problem
  → [VMB Retrieval] → memory hints
  → [LTG] small model decomposes → atomic primitives
  → [RGC] build directed reasoning graph
  → [VCE] rule-check → small model verify → LLM escalate (Qwen2.5-72B)
  → [Deliberate] backtrack + branch on failures
  → [Final Gen] small model synthesizes answer from verified graph
  → [VMB Store] save successful patterns
```

## Comparison
- **Baseline A**: Zero-Shot  
- **Baseline B**: Chain-of-Thought  
- **SLD-VM**: Full pipeline

---
> **Kaggle**: Tesla P100 16GB | Phi-4-mini-instruct (local) | Qwen2.5-72B (HF API)

In [1]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Install packages                                  ║
# ╚══════════════════════════════════════════════════════════════╝
# !pip install -q transformers datasets accelerate huggingface_hub tqdm
print('✅ Packages ready')

✅ Packages ready


# CELL 2 — HuggingFace Login 

In [2]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2 — HuggingFace Login                                ║
# ╚══════════════════════════════════════════════════════════════╝
from huggingface_hub import login
HF_TOKEN = ""   # ← paste your HF token here
login(HF_TOKEN)
print('✅ HuggingFace login done')

✅ HuggingFace login done


# CELL 3 — Imports & Global Config 

In [3]:

import torch, json, re, random, time, hashlib, gc
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Tuple, Any
from collections import defaultdict, Counter
from enum import Enum
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import InferenceClient

CFG = {
    'small_model_id':    'microsoft/Phi-4-mini-instruct',
    'verifier_model':    'Qwen/Qwen2.5-72B-Instruct',
    'llm_conf_threshold': 0.95,
    'n_samples':         10,
    'seed':              42,
    'max_branches':      3,
    'max_depth':         7,
    'sc_n':              3,
    'vmb_max_size':      500,
    'vmb_top_k':         3,
    'api_sleep_sec':     3,
}

device = 'cuda' if torch.cuda.is_available() else 'cpu'
random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])

STATS = {'llm_calls':0,'llm_errors':0,'backtracks':0,'vmb_hits':0,'nodes_pruned':0}

print(f'Device : {device}')
if device == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print('✅ Config ready')

Device : cuda
GPU    : Tesla P100-PCIE-16GB
VRAM   : 17.1 GB
✅ Config ready


# CELL 4 — Load Small Model (Phi-4-mini-instruct)

In [4]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Load Small Model (Phi-4-mini-instruct)           ║
# ╚══════════════════════════════════════════════════════════════╝
print(f'Loading {CFG["small_model_id"]} ...')
tokenizer = AutoTokenizer.from_pretrained(CFG['small_model_id'])
small_model = AutoModelForCausalLM.from_pretrained(
    CFG['small_model_id'], device_map='auto', dtype=torch.float16)
small_model.eval()
print('✅ Model loaded')
print(f'   VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB')

def _build_chat(system, user):
    msgs = []
    if system: msgs.append({'role':'system','content':system})
    msgs.append({'role':'user','content':user})
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def small_gen(system, user, max_new_tokens=400, temperature=0.0, do_sample=False):
    prompt = _build_chat(system, user)
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    L = inputs['input_ids'].shape[1]
    with torch.no_grad():
        out = small_model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature if do_sample else None,
            pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][L:], skip_special_tokens=True).strip()

def small_gen_with_conf(system, user, max_new_tokens=400):
    prompt = _build_chat(system, user)
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    L = inputs['input_ids'].shape[1]
    with torch.no_grad():
        out = small_model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            return_dict_in_generate=True, output_scores=True)
    scores = out.scores
    conf = (sum(torch.softmax(s[0],dim=-1).max().item() for s in scores)/len(scores)
            if scores else 0.5)
    text = tokenizer.decode(out.sequences[0][L:], skip_special_tokens=True).strip()
    return text, conf

def small_gen_n(system, user, n=3, temperature=0.7, max_new_tokens=300):
    return [small_gen(system, user, max_new_tokens=max_new_tokens,
                      temperature=temperature, do_sample=True) for _ in range(n)]

test = small_gen('You are a math solver.', 'What is 12 x 7?', max_new_tokens=30)
print(f'\n✅ Sanity check: "{test}"')

Loading microsoft/Phi-4-mini-instruct ...


config.json: 0.00B [00:00, ?B/s]

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

✅ Model loaded
   VRAM used: 7.7 GB

✅ Sanity check: "12 multiplied by 7 equals 84."


# CELL 5 — LLM Verifier (Qwen2.5-72B via HF API)

In [5]:
client = InferenceClient(token=HF_TOKEN)
LLM_AVAILABLE = False

try:
    resp = client.chat_completion(
        model=CFG['verifier_model'],
        messages=[{'role':'user','content':'Reply with just: OK'}],
        max_tokens=5)
    print(f'✅ LLM Verifier ready | test: {resp.choices[0].message.content.strip()}')
    LLM_AVAILABLE = True
except Exception as e:
    print(f'⚠️  LLM unavailable: {e}')
    print('   Pipeline will use rule-based + small model VCE only')

def llm_call(system, user, max_tokens=200, retries=3):
    if not LLM_AVAILABLE: return None
    for attempt in range(retries):
        try:
            time.sleep(CFG['api_sleep_sec'])
            resp = client.chat_completion(
                model=CFG['verifier_model'],
                messages=[{'role':'system','content':system},
                          {'role':'user','content':user}],
                max_tokens=max_tokens)
            STATS['llm_calls'] += 1
            return resp.choices[0].message.content.strip()
        except Exception as e:
            STATS['llm_errors'] += 1
            if attempt < retries-1: time.sleep(5*(attempt+1))
    return None

print(f'LLM status: {"✅ Active" if LLM_AVAILABLE else "⚠️  Fallback only"}')

✅ LLM Verifier ready | test: OK
LLM status: ✅ Active


#  CELL 6 — Load Hard Datasets 

In [12]:

# ── MATH (Hendrycks Competition Math) ─────────────────────────────────────────
print('Loading MATH dataset...')

# chiayewken/competition_math
# hendrycks/competition_math

math_raw = load_dataset('chiayewken/competition_math', split='test')
math_filtered = [s for s in math_raw
                 if s.get('level') in ['Level 3','Level 4']
                 and s.get('type') in ['Algebra','Number Theory','Counting & Probability']]
random.shuffle(math_filtered)
math_samples = math_filtered[:CFG['n_samples']]
print(f'✅ MATH: {len(math_samples)} samples (Level 3-4)')
print(f'   Example: {math_samples[0]["problem"][:100]}...')

# ── ARC-Challenge (AI2 Reasoning Challenge) ────────────────────────────────────
# Loaded from allenai/ai2_arc — reliable HuggingFace dataset, no login needed
# Hard science MCQ: requires multi-step reasoning, not just recall
# Small models score ~45-65%; random baseline = 25%
print('\nLoading ARC-Challenge dataset...')
arc_raw = load_dataset('allenai/ai2_arc', 'ARC-Challenge', split='test')
arc_list = list(arc_raw)
random.shuffle(arc_list)
arc_samples = arc_list[:CFG['n_samples']]
print(f'✅ ARC-Challenge: {len(arc_samples)} samples')
print(f'   Example Q: {arc_samples[0]["question"][:100]}')
print(f'   Choices  : {arc_samples[0]["choices"]["text"]}')
print(f'   Labels   : {arc_samples[0]["choices"]["label"]}')
print(f'   Answer   : {arc_samples[0]["answerKey"]}')
print('\n✅ Both datasets loaded')

Loading MATH dataset...
✅ MATH: 10 samples (Level 3-4)
   Example: Amy was born on a Tuesday. What is the probability that exactly two of her three best friends were a...

Loading ARC-Challenge dataset...
✅ ARC-Challenge: 10 samples
   Example Q: Which of the following could occur as a result of decreasing the heat energy of a gas?
   Choices  : ['condensation', 'evaporation', 'radiation', 'vaporization']
   Labels   : ['A', 'B', 'C', 'D']
   Answer   : A

✅ Both datasets loaded


# CELL 7 — Answer Extractors & Correctness Checkers 

In [30]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 7 — Answer Extractors & Correctness Checkers         ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Nested brace extractor (fixes \boxed{\frac{18}{77}} truncation) ───────────
def _extract_boxed(text):
    idx = text.find('\\boxed{')
    if idx == -1:
        return None
    start = idx + len('\\boxed{')
    depth, i = 1, start
    while i < len(text) and depth > 0:
        if text[i] == '{':
            depth += 1
        elif text[i] == '}':
            depth -= 1
        i += 1
    return text[start:i-1].strip() if depth == 0 else None

# ── MATH extractors ───────────────────────────────────────────────────────────
def extract_math_gold(solution):
    ans = _extract_boxed(solution)
    if ans:
        return ans
    nums = re.findall(r'-?\d+(?:\.\d+)?', solution)
    return nums[-1] if nums else None

def extract_math_pred(text):
    ans = _extract_boxed(text)
    if ans:
        return ans
    for pat in [r'(?:answer|result|therefore|thus)[:\s]+([\-\d\/\.\,\s]+)',
                r'=\s*([\-\d\/\.]+)\s*$',
                r'####\s*([\-\d\/\.]+)']:
        m = re.search(pat, text, re.IGNORECASE|re.MULTILINE)
        if m:
            val = m.group(1).strip().rstrip('.')
            if re.search(r'\d', val):
                return val
    nums = re.findall(r'-?\d+(?:/\d+)?(?:\.\d+)?', text)
    return nums[-1] if nums else None

def normalize_math(s):
    if s is None:
        return ''
    s = re.sub(r'[\$\\,\s]', '', s.strip().lower())
    s = s.replace('dfrac','frac').replace('tfrac','frac')
    try:
        return str(round(float(s), 4))
    except:
        return s

def math_correct(pred, gold):
    if pred is None or gold is None:
        return False
    if normalize_math(pred) == normalize_math(gold):
        return True
    try:
        return abs(float(pred.replace(',','')) - float(gold.replace(',',''))) < 0.01
    except:
        return False

# ── ARC-Challenge extractors ──────────────────────────────────────────────────
def format_arc(sample):
    choices = sample['choices']
    opts = '\n'.join(f'{choices["label"][i]}) {choices["text"][i]}'
                      for i in range(len(choices['text'])))
    return f"{sample['question']}\n\nOptions:\n{opts}"

def arc_gold(sample):
    key = str(sample['answerKey']).strip().upper()
    return {'1':'A','2':'B','3':'C','4':'D'}.get(key, key)

def extract_arc_pred(text):
    text = text.strip()
    m = re.search(r'(?:answer(?:\s+is)?|therefore|thus|correct(?:\s+answer)?(?:\s+is)?)[:\s]+([A-D])',
                  text, re.IGNORECASE)
    if m:
        return m.group(1).upper()
    m = re.search(r'[\(\[]([A-D])[\)\]]', text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r'\b([A-D])\b', text)
    return matches[-1].upper() if matches else None

def arc_correct(pred, gold):
    if pred is None:
        return False
    return pred.strip().upper() == gold.strip().upper()

# Self-tests
assert _extract_boxed('The answer is \\boxed{\\frac{18}{77}}') == '\\frac{18}{77}'
assert extract_math_pred('The answer is \\boxed{42}') == '42'
assert math_correct('42', '42.0')
assert extract_arc_pred('The correct answer is B.') == 'B'
assert extract_arc_pred('Answer: (C)') == 'C'
assert arc_gold({'answerKey': '2'}) == 'B'
print('✅ All extractors verified')

✅ All extractors verified


# CELL 8 — Reasoning Graph Data Structures

In [31]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 8 — Reasoning Graph Data Structures                  ║
# ╚══════════════════════════════════════════════════════════════╝

class NodeStatus(Enum):
    PENDING = 'pending'
    VALID   = 'valid'
    INVALID = 'invalid'
    PRUNED  = 'pruned'

@dataclass
class RNode:
    node_id:    str
    content:    str
    node_type:  str
    parents:    List[str] = field(default_factory=list)
    children:   List[str] = field(default_factory=list)
    status:     NodeStatus = NodeStatus.PENDING
    confidence: float = 0.5
    depth:      int = 0
    branch_id:  int = 0
    vce_reason: str = ''

@dataclass
class ReasoningGraph:
    problem: str
    task:    str
    nodes:   Dict[str, RNode] = field(default_factory=dict)
    root_id: Optional[str] = None

    def add_node(self, content, node_type, parents=None, depth=0, branch_id=0):
        nid = f'N{len(self.nodes):03d}'
        node = RNode(node_id=nid, content=content, node_type=node_type,
                     parents=parents or [], depth=depth, branch_id=branch_id)
        self.nodes[nid] = node
        for pid in (parents or []):
            if pid in self.nodes: self.nodes[pid].children.append(nid)
        if self.root_id is None: self.root_id = nid
        return nid

    def prune_subtree(self, node_id):
        if node_id not in self.nodes: return
        self.nodes[node_id].status = NodeStatus.PRUNED
        STATS['nodes_pruned'] += 1
        for child in self.nodes[node_id].children:
            self.prune_subtree(child)

    def valid_nodes(self):
        return [n for n in self.nodes.values() if n.status == NodeStatus.VALID]

    def valid_path_to(self, node_id):
        path, node = [], self.nodes.get(node_id)
        while node:
            path.insert(0, node.content)
            parent_ids = [p for p in node.parents
                          if p in self.nodes and self.nodes[p].status == NodeStatus.VALID]
            node = self.nodes[parent_ids[-1]] if parent_ids else None
        return path

    def summary(self):
        c = Counter(n.status.value for n in self.nodes.values())
        return (f'Graph({len(self.nodes)} nodes): '
                f'✅{c["valid"]} ❌{c["invalid"]} ✂️{c["pruned"]} ⏳{c["pending"]}')

print('✅ Graph data structures ready')

✅ Graph data structures ready


# CELL 9 — LTG: Latent Thought Generation

In [32]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 9 — LTG: Latent Thought Generation                   ║
# ╚══════════════════════════════════════════════════════════════╝

LTG_SYS_MATH = """You are a step-by-step math reasoner.
Decompose the problem into numbered atomic steps.
Each step is ONE simple claim, fact, or arithmetic operation.
Format STRICTLY:
Step 1: [premise / given fact]
Step 2: [single deduction or calculation]
...
Final: [the answer]"""

LTG_SYS_ARC = """You are a science reasoning decomposer.
Break the question into numbered atomic reasoning steps.
Each step is ONE scientific fact, or eliminates one wrong option with a reason.
Format STRICTLY:
Step 1: [identify what the question asks]
Step 2: [relevant scientific fact]
Step 3: [eliminate wrong options with reasons]
Final: [A, B, C, or D]"""

def ltg_decompose(problem, task, memory_hint=''):
    sys = LTG_SYS_MATH if task == 'math' else LTG_SYS_ARC
    user = f'{memory_hint}\n\nProblem:\n{problem}' if memory_hint else f'Problem:\n{problem}'
    text, conf = small_gen_with_conf(sys, user, max_new_tokens=400)
    primitives = []
    for line in text.split('\n'):
        m = re.match(r'^(?:Step\s*\d+|Final)[:\.] *(.+)', line.strip(), re.IGNORECASE)
        if m:
            content = m.group(1).strip()
            if content: primitives.append(content)
    if len(primitives) < 2:
        primitives = [s.strip() for s in re.split(r'[.\n]', text)
                      if len(s.strip()) > 8][:CFG['max_depth']]
    return primitives[:CFG['max_depth']], conf

# Smoke test
_prims, _conf = ltg_decompose(arc_samples[0]['question']+'\n'+str(arc_samples[0]['choices']), 'arc')
print(f'✅ LTG ready | conf={_conf:.2f} | {len(_prims)} primitives')
for i,p in enumerate(_prims): print(f'  [{i}] {p}')

✅ LTG ready | conf=0.86 | 4 primitives
  [0] Identify what the question asks
  [1] Relevant scientific fact
  [2] Eliminate wrong options with reasons
  [3] A (condensation)


# CELL 10 — RGC: Reasoning Graph Construction

In [33]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 10 — RGC: Reasoning Graph Construction               ║
# ╚══════════════════════════════════════════════════════════════╝

def rgc_build(problem, task, primitives, branch_id=0):
    graph = ReasoningGraph(problem=problem, task=task)
    if not primitives: return graph
    prev_id = None
    for depth, prim in enumerate(primitives):
        ntype = ('premise' if depth == 0
                 else 'conclusion' if depth == len(primitives)-1
                 else 'deduction')
        nid = graph.add_node(prim, ntype,
                             parents=[prev_id] if prev_id else [],
                             depth=depth, branch_id=branch_id)
        prev_id = nid
    return graph

def rgc_add_branch(graph, anchor_id, alt_steps, branch_id):
    new_ids, prev_id = [], anchor_id
    anchor_depth = graph.nodes[anchor_id].depth
    for i, step in enumerate(alt_steps):
        depth = anchor_depth + i + 1
        ntype = 'conclusion' if i == len(alt_steps)-1 else 'deduction'
        nid = graph.add_node(step, ntype, parents=[prev_id],
                             depth=depth, branch_id=branch_id)
        new_ids.append(nid)
        prev_id = nid
    return new_ids

print('✅ RGC ready')

✅ RGC ready


# CELL 11 — VCE: 3-Level Verifier / Consistency Engine

In [34]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 11 — VCE: 3-Level Verifier / Consistency Engine      ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Level 1: Rule-based (free, instant) ───────────────────────────────────────
def rule_arithmetic(content):
    pattern = r'(-?\d+\.?\d*)\s*([+\-×x*/÷])\s*(-?\d+\.?\d*)\s*=\s*(-?\d+\.?\d*)'
    op_map = {'+': lambda a,b: a+b, '-': lambda a,b: a-b,
              '*': lambda a,b: a*b, 'x': lambda a,b: a*b, '×': lambda a,b: a*b,
              '/': lambda a,b: a/b if b!=0 else None, '÷': lambda a,b: a/b if b!=0 else None}
    for a_s, op, b_s, r_s in re.findall(pattern, content):
        try:
            a, b, claimed = float(a_s), float(b_s), float(r_s)
            fn = op_map.get(op)
            if fn:
                actual = fn(a, b)
                if actual is not None and abs(actual-claimed) > 0.01:
                    return False, f'Arithmetic error: {a}{op}{b}={claimed} (correct={actual:.4f})'
        except: continue
    return True, 'ok'

def rule_contradiction(content, priors):
    established = {}
    for node in priors:
        for var, val in re.findall(r'(\w+)\s*=\s*(-?\d+(?:\.\d+)?)', node.content):
            established[var.lower()] = float(val)
    for var, val in re.findall(r'(\w+)\s*=\s*(-?\d+(?:\.\d+)?)', content):
        if var.lower() in established:
            if abs(established[var.lower()] - float(val)) > 0.01:
                return False, f'Contradiction: {var}={val} vs earlier {var}={established[var.lower()]}'
    return True, 'ok'

def rule_arc_option(content, sample):
    m = re.search(r'(?:answer|choose|option|therefore)[:\s]+([A-D])', content, re.IGNORECASE)
    if m:
        label = m.group(1).upper()
        valid = [l.upper() for l in sample['choices']['label']]
        if label not in valid:
            return False, f'Option {label} not in valid choices {valid}'
    return True, 'ok'

# ── Level 2: Small model verification ────────────────────────────────────────
VCE_SM_SYS = """You are a strict logical verifier.
Given a reasoning step and its context, determine if the step is VALID or INVALID.
Reply ONLY with:
VERDICT: VALID
or
VERDICT: INVALID
REASON: one sentence"""

def small_verify(problem, step, prior_nodes):
    prior_text = ' → '.join(n.content for n in prior_nodes[-3:]) or 'none'
    user = (f'Problem: {problem[:300]}\n'
            f'Prior steps: {prior_text}\n'
            f'Step to verify: {step}')
    text, conf = small_gen_with_conf(VCE_SM_SYS, user, max_new_tokens=80)
    is_valid = not re.search(r'VERDICT:\s*INVALID', text, re.IGNORECASE)
    return is_valid, conf, text[:120]

# ── Level 3: LLM escalation ───────────────────────────────────────────────────
VCE_LLM_SYS = """You are an expert verifier for mathematical and scientific reasoning.
Flag any factual error, arithmetic mistake, or logical contradiction.
Reply EXACTLY:
VERDICT: VALID or INVALID
REASON: one sentence"""

def llm_verify(problem, step, prior_nodes):
    prior_text = ' → '.join(n.content for n in prior_nodes[-4:]) or 'none'
    user = (f'Problem: {problem[:400]}\nPrior steps: {prior_text}\nStep: {step}')
    resp = llm_call(VCE_LLM_SYS, user, max_tokens=150)
    if resp is None: return True, 'LLM unavailable — pass'
    is_valid = not re.search(r'VERDICT:\s*INVALID', resp, re.IGNORECASE)
    m = re.search(r'REASON:\s*(.+)', resp, re.IGNORECASE)
    reason = m.group(1).strip()[:100] if m else resp[:100]
    return is_valid, f'[LLM] {reason}'

# ── Combined VCE ──────────────────────────────────────────────────────────────
def vce_verify_node(graph, node_id, sample=None):
    node = graph.nodes[node_id]
    priors = [graph.nodes[p] for p in node.parents
              if p in graph.nodes and graph.nodes[p].status == NodeStatus.VALID]
    ok, reason = rule_arithmetic(node.content)
    if not ok: return False, f'[RULE-ARITH] {reason}'
    ok, reason = rule_contradiction(node.content, priors)
    if not ok: return False, f'[RULE-CONTRA] {reason}'
    if graph.task == 'arc' and sample:
        ok, reason = rule_arc_option(node.content, sample)
        if not ok: return False, f'[RULE-OPTION] {reason}'
    if node.node_type == 'premise':
        node.confidence = 0.9
        return True, '[PREMISE] accepted'
    sm_valid, sm_conf, sm_reason = small_verify(graph.problem, node.content, priors)
    node.confidence = sm_conf
    if sm_conf < CFG['llm_conf_threshold'] and LLM_AVAILABLE:
        return llm_verify(graph.problem, node.content, priors)
    return sm_valid, f'[SM|conf={sm_conf:.2f}] {sm_reason[:80]}'

def vce_run_graph(graph, sample=None):
    for node in sorted(graph.nodes.values(), key=lambda n: n.depth):
        if node.status != NodeStatus.PENDING: continue
        if any(graph.nodes[p].status in [NodeStatus.PRUNED, NodeStatus.INVALID]
               for p in node.parents if p in graph.nodes):
            graph.prune_subtree(node.node_id); continue
        is_valid, reason = vce_verify_node(graph, node.node_id, sample)
        node.vce_reason = reason
        node.status = NodeStatus.VALID if is_valid else NodeStatus.INVALID
        if not is_valid: graph.prune_subtree(node.node_id)
    return graph

print('✅ VCE (3-level) ready')

✅ VCE (3-level) ready


# CELL 12 — VMB: Verifiable Memory Bank

In [35]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 12 — VMB: Verifiable Memory Bank                     ║
# ╚══════════════════════════════════════════════════════════════╝

@dataclass
class MemoryPattern:
    pid: str; task: str; problem_hash: str
    keywords: List[str]; primitives: List[str]; answer: str
    reliability: float = 1.0; usage_count: int = 0

class VMB:
    def __init__(self, max_size=500):
        self.patterns: Dict[str, MemoryPattern] = {}
        self.max_size = max_size
        self._kw_idx: Dict[str, List[str]] = defaultdict(list)

    def _hash(self, text):
        return hashlib.md5(text.strip().lower().encode()).hexdigest()[:10]

    def _keywords(self, text, task):
        text = text.lower()
        pools = {
            'math': ['factor','divisor','prime','integer','sum','product','remainder',
                     'square','cube','root','fraction','probability','combinat','permut'],
            'arc':  ['because','causes','result','energy','force','atom','cell',
                     'photosynthesis','gravity','chemical','physical','organism',
                     'process','system','temperature','pressure','evolution']
        }
        return [kw for kw in pools.get(task,[]) if kw in text]

    def store(self, problem, task, primitives, answer):
        if len(self.patterns) >= self.max_size:
            worst = min(self.patterns.values(), key=lambda p: p.reliability)
            for kw in worst.keywords:
                self._kw_idx[kw] = [x for x in self._kw_idx[kw] if x != worst.pid]
            del self.patterns[worst.pid]
        phash = self._hash(problem)
        pid = f'P{len(self.patterns):04d}'
        kws = self._keywords(problem, task)
        pat = MemoryPattern(pid=pid, task=task, problem_hash=phash,
                            keywords=kws, primitives=primitives, answer=answer)
        self.patterns[pid] = pat
        for kw in kws: self._kw_idx[kw].append(pid)

    def retrieve(self, problem, task, top_k=3):
        kws = self._keywords(problem, task)
        scores: Dict[str, float] = defaultdict(float)
        for kw in kws:
            for pid in self._kw_idx.get(kw,[]):
                if pid in self.patterns and self.patterns[pid].task == task:
                    p = self.patterns[pid]
                    scores[pid] += p.reliability * (1 + 0.05*p.usage_count)
        top = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
        result = []
        for pid,_ in top:
            if pid in self.patterns:
                self.patterns[pid].usage_count += 1
                result.append(self.patterns[pid])
        if result: STATS['vmb_hits'] += 1
        return result

    def update_reliability(self, problem, success):
        phash = self._hash(problem)
        for p in self.patterns.values():
            if p.problem_hash == phash:
                p.reliability = (min(1.0, p.reliability*1.15) if success
                                 else max(0.05, p.reliability*0.75))

    def stats(self):
        if not self.patterns: return 'VMB: empty'
        avg_r = sum(p.reliability for p in self.patterns.values())/len(self.patterns)
        return f'VMB: {len(self.patterns)} patterns | avg_rel={avg_r:.2f} | hits={STATS["vmb_hits"]}'

vmb = VMB(max_size=CFG['vmb_max_size'])
print('✅ VMB ready')

✅ VMB ready


# CELL 13 — Deliberation Engine

In [36]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 13 — Deliberation Engine                             ║
# ╚══════════════════════════════════════════════════════════════╝

REGEN_SYS_MATH = """A reasoning step was rejected as invalid.
Given the valid steps so far and the failed step, suggest 2-3 ALTERNATIVE
next steps to continue the reasoning toward the answer.
Format:
Alt 1: [step]
Alt 2: [step]
Alt 3: [step]"""

REGEN_SYS_ARC = """A reasoning step about this science question was rejected.
Suggest 2 ALTERNATIVE reasoning steps from the valid steps so far.
Focus on scientific facts and eliminating wrong answer choices.
Format:
Alt 1: [step]
Alt 2: [step]"""

FINAL_SYS_MATH = """Given verified reasoning steps, state the final answer.
Put the answer inside \\boxed{} only. Nothing else."""

FINAL_SYS_ARC = """Given verified reasoning steps about this science question,
state which option (A, B, C, or D) is correct.
Output ONLY the single letter. Nothing else."""

def deliberate_alternatives(problem, task, valid_steps, failed_step):
    sys = REGEN_SYS_MATH if task == 'math' else REGEN_SYS_ARC
    valid_text = ' → '.join(valid_steps[-3:]) if valid_steps else 'none yet'
    user = (f'Problem: {problem[:400]}\n'
            f'Valid steps so far: {valid_text}\n'
            f'Failed step: {failed_step}\nSuggest alternatives:')
    text = small_gen(sys, user, max_new_tokens=200)
    alts = []
    for line in text.split('\n'):
        m = re.match(r'Alt\s*\d+:\s*(.+)', line.strip(), re.IGNORECASE)
        if m: alts.append(m.group(1).strip())
    return alts[:CFG['max_branches']]

def deliberate_extract_answer(graph, task, sample=None):
    valid = graph.valid_nodes()
    if not valid: return None
    # Try deepest conclusion node first
    for node in sorted(valid, key=lambda n: n.depth, reverse=True):
        if node.node_type == 'conclusion':
            ans = extract_math_pred(node.content) if task=='math' else extract_arc_pred(node.content)
            if ans: return ans
    # Fallback: synthesize from all valid steps
    steps = ' → '.join(n.content for n in sorted(valid, key=lambda n: n.depth))
    if task == 'math':
        out = small_gen(FINAL_SYS_MATH,
                        f'Problem: {graph.problem[:400]}\nVerified steps: {steps}',
                        max_new_tokens=50)
        return extract_math_pred(out)
    else:
        choices = sample['choices'] if sample else {'text':[],'label':[]}
        opts = '\n'.join(f'{choices["label"][i]}) {choices["text"][i]}'
                         for i in range(len(choices['text'])))
        out = small_gen(FINAL_SYS_ARC,
                        f'Question: {graph.problem[:300]}\nOptions:\n{opts}\nVerified steps: {steps}',
                        max_new_tokens=20)
        return extract_arc_pred(out)

print('✅ Deliberation engine ready')

✅ Deliberation engine ready


# CELL 14 — Full SLD-VM Pipeline

In [37]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 14 — Full SLD-VM Pipeline                            ║
# ╚══════════════════════════════════════════════════════════════╝

@dataclass
class PipelineResult:
    answer:     Optional[str]
    graph:      ReasoningGraph
    used_vmb:   bool  = False
    backtracks: int   = 0
    llm_calls:  int   = 0
    conf:       float = 0.0

def sldvm(problem, task, sample=None, verbose=False):
    """Full SLD-VM: VMB → LTG → RGC → VCE → Deliberate → Answer → VMB Store"""
    pre_llm = STATS['llm_calls']
    backtracks = 0

    # Stage 1: VMB Retrieval
    patterns = vmb.retrieve(problem, task, top_k=CFG['vmb_top_k'])
    used_vmb = len(patterns) > 0
    memory_hint = ''
    if patterns:
        hint = ' → '.join(patterns[0].primitives[:3])
        memory_hint = f'[Similar problem solved via: {hint}]'
        if verbose: print(f'  📚 VMB: {len(patterns)} patterns retrieved')

    # Stage 2: LTG
    primitives, ltg_conf = ltg_decompose(problem, task, memory_hint)
    if verbose: print(f'  🔨 LTG: {len(primitives)} primitives | conf={ltg_conf:.2f}')
    if not primitives:
        return PipelineResult(answer=None, graph=ReasoningGraph(problem=problem, task=task),
                              used_vmb=used_vmb)

    # Stage 3: RGC
    graph = rgc_build(problem, task, primitives, branch_id=0)

    # Stage 4: VCE
    graph = vce_run_graph(graph, sample)
    if verbose: print(f'  ✅ VCE: {graph.summary()}')

    # Stage 5: Deliberation — backtrack on invalid nodes
    invalid_nodes = [n for n in graph.nodes.values() if n.status == NodeStatus.INVALID]
    for inv_node in invalid_nodes[:CFG['max_branches']]:
        if backtracks >= CFG['max_branches']: break
        valid_ancestors = [graph.nodes[p] for p in inv_node.parents
                           if p in graph.nodes and graph.nodes[p].status == NodeStatus.VALID]
        if not valid_ancestors: continue
        anchor = valid_ancestors[-1]
        valid_path = graph.valid_path_to(anchor.node_id)
        alts = deliberate_alternatives(problem, task, valid_path, inv_node.content)
        if alts:
            backtracks += 1
            STATS['backtracks'] += 1
            new_ids = rgc_add_branch(graph, anchor.node_id, alts, branch_id=backtracks)
            for nid in new_ids:
                if graph.nodes[nid].status == NodeStatus.PENDING:
                    ok, reason = vce_verify_node(graph, nid, sample)
                    graph.nodes[nid].vce_reason = reason
                    graph.nodes[nid].status = NodeStatus.VALID if ok else NodeStatus.INVALID
            if verbose: print(f'  🔄 Backtrack #{backtracks}: {len(alts)} alternatives')

    # Stage 6: Extract answer
    answer = deliberate_extract_answer(graph, task, sample)

    # Stage 7: VMB Store
    if answer is not None:
        valid_prims = [n.content for n in sorted(graph.valid_nodes(), key=lambda n: n.depth)]
        if valid_prims: vmb.store(problem, task, valid_prims, answer)

    valid = graph.valid_nodes()
    conf = sum(n.confidence for n in valid)/len(valid) if valid else 0.0

    return PipelineResult(answer=answer, graph=graph, used_vmb=used_vmb,
                          backtracks=backtracks,
                          llm_calls=STATS['llm_calls']-pre_llm, conf=conf)

# Smoke test
print('Running smoke test on ARC...')
_s = arc_samples[0]
_r = sldvm(format_arc(_s), 'arc', sample=_s, verbose=True)
print(f'\n✅ Smoke test: pred={_r.answer} gold={arc_gold(_s)}')
print(f'   {_r.graph.summary()}')

Running smoke test on ARC...
  🔨 LTG: 6 primitives | conf=0.86
  ✅ VCE: Graph(6 nodes): ✅6 ❌0 ✂️0 ⏳0

✅ Smoke test: pred=A gold=A
   Graph(6 nodes): ✅6 ❌0 ✂️0 ⏳0


#  CELL 15 — Baseline A: Zero-Shot

In [38]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 15 — Baseline A: Zero-Shot                           ║
# ╚══════════════════════════════════════════════════════════════╝

RESULTS = {'MATH': {}, 'ARC': {}}

ZS_SYS_MATH = 'You are a math solver. Solve and put your final answer inside \\boxed{}.'
ZS_SYS_ARC  = ('You are a science expert. '
                'Output ONLY the letter (A, B, C, or D) of the correct answer.')

# ── MATH Zero-Shot ─────────────────────────────────────────────────────────────
correct, null_count = 0, 0
zs_math_records = []
for i, s in enumerate(tqdm(math_samples, desc='MATH Zero-Shot')):
    out  = small_gen(ZS_SYS_MATH, s['problem'], max_new_tokens=400)
    pred = extract_math_pred(out)
    gold = extract_math_gold(s['solution'])
    ok   = math_correct(pred, gold)
    if ok: correct += 1
    if pred is None: null_count += 1
    zs_math_records.append({'out':out,'pred':pred,'gold':gold,'ok':ok,'s':s})
    if i < 3: print(f'[{i+1}] pred={pred} gold={gold} ok={ok} | {out[:100]}')
RESULTS['MATH']['zero_shot'] = correct / len(math_samples)
print(f'\n✅ MATH Zero-Shot: {RESULTS["MATH"]["zero_shot"]:.1%} | Null={null_count}')

# ── ARC Zero-Shot ──────────────────────────────────────────────────────────────
correct, null_count = 0, 0
zs_arc_records = []
for i, s in enumerate(tqdm(arc_samples, desc='ARC Zero-Shot')):
    problem = format_arc(s)
    out  = small_gen(ZS_SYS_ARC, problem, max_new_tokens=50)
    pred = extract_arc_pred(out)
    gold = arc_gold(s)
    ok   = arc_correct(pred, gold)
    if ok: correct += 1
    if pred is None: null_count += 1
    zs_arc_records.append({'out':out,'pred':pred,'gold':gold,'ok':ok,'s':s})
    if i < 3: print(f'[{i+1}] pred={pred} gold={gold} ok={ok} | {out[:80]}')
RESULTS['ARC']['zero_shot'] = correct / len(arc_samples)
print(f'\n✅ ARC Zero-Shot: {RESULTS["ARC"]["zero_shot"]:.1%} | Null={null_count} | (random=25%)')

MATH Zero-Shot:  10%|█         | 1/10 [00:18<02:43, 18.16s/it]

[1] pred=3 gold=\frac{18}{343} ok=False | To determine the probability that exactly two out of Amy's three best friends were born on a Tuesday


MATH Zero-Shot:  20%|██        | 2/10 [00:34<02:15, 16.99s/it]

[2] pred=12 gold=12 ok=True | We are given the equations:
\[ 3n + m = 14 \]
\[ n + m = 1 \]

First, we will solve for \( m \) in t


MATH Zero-Shot:  30%|███       | 3/10 [00:52<02:02, 17.47s/it]

...

[2] pred=C gold=C ok=True | C) As the temperature increases, the helium in the balloons expands.


ARC Zero-Shot:  30%|███       | 3/10 [00:01<00:03,  1.88it/s]

[3] pred=C gold=C ok=True | C) cutting down trees to build houses


ARC Zero-Shot: 100%|██████████| 10/10 [00:05<00:00,  1.92it/s]


✅ ARC Zero-Shot: 100.0% | Null=0 | (random=25%)


# CELL 16 — Baseline B: Chain-of-Thought

In [39]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 16 — Baseline B: Chain-of-Thought                    ║
# ╚══════════════════════════════════════════════════════════════╝

COT_SYS_MATH = """You are an expert math solver.
Think step by step. Put your final answer inside \\boxed{} on its own line."""

COT_SYS_ARC = """You are a science expert.
Reason step by step. Eliminate wrong answers. End with: Answer: [A/B/C/D]"""

MATH_COT_EX = """Example:
Problem: How many primes between 10 and 30?
Step 1: Check 11: prime ✓  13: prime ✓  17: prime ✓  19: prime ✓  23: prime ✓  29: prime ✓
Step 2: Count = 6
\\boxed{6}"""

ARC_COT_EX = """Example:
Which process do plants use to make food?
A) Respiration  B) Photosynthesis  C) Digestion  D) Fermentation
Step 1: Plants convert sunlight + CO2 + water into glucose.
Step 2: That process is photosynthesis. Respiration/digestion/fermentation don't produce food.
Answer: B"""

# ── MATH CoT ───────────────────────────────────────────────────────────────────
correct, null_count = 0, 0
cot_math_records = []
for i, s in enumerate(tqdm(math_samples, desc='MATH CoT')):
    out  = small_gen(COT_SYS_MATH, f'{MATH_COT_EX}\n\nProblem: {s["problem"]}', max_new_tokens=500)
    pred = extract_math_pred(out)
    gold = extract_math_gold(s['solution'])
    ok   = math_correct(pred, gold)
    if ok: correct += 1
    if pred is None: null_count += 1
    cot_math_records.append({'out':out,'pred':pred,'gold':gold,'ok':ok,'s':s})
    if i < 3: print(f'[{i+1}] pred={pred} gold={gold} ok={ok} | {out[:150]}')
RESULTS['MATH']['cot'] = correct / len(math_samples)
print(f'\n✅ MATH CoT: {RESULTS["MATH"]["cot"]:.1%} | Null={null_count}')

# ── ARC CoT ────────────────────────────────────────────────────────────────────
correct, null_count = 0, 0
cot_arc_records = []
for i, s in enumerate(tqdm(arc_samples, desc='ARC CoT')):
    problem = format_arc(s)
    out  = small_gen(COT_SYS_ARC, f'{ARC_COT_EX}\n\n{problem}', max_new_tokens=300)
    pred = extract_arc_pred(out)
    gold = arc_gold(s)
    ok   = arc_correct(pred, gold)
    if ok: correct += 1
    if pred is None: null_count += 1
    cot_arc_records.append({'out':out,'pred':pred,'gold':gold,'ok':ok,'s':s})
    if i < 3: print(f'[{i+1}] pred={pred} gold={gold} ok={ok} | {out[:120]}')
RESULTS['ARC']['cot'] = correct / len(arc_samples)
print(f'\n✅ ARC CoT: {RESULTS["ARC"]["cot"]:.1%} | Null={null_count}')

print('\n── Baseline Summary ─────────────────────────────────────')
for ds in ['MATH','ARC']:
    for k,v in RESULTS[ds].items():
        print(f'  {ds:6s} {k:12s}: {v:.1%}')

MATH CoT:  10%|█         | 1/10 [00:05<00:52,  5.86s/it]

[1] pred=\frac{18}{343} gold=\frac{18}{343} ok=True | Step 1: Calculate the probability that one friend was born on a Tuesday: 1/7
Step 2: Calculate the probability that one friend was not born on a Tuesd


MATH CoT:  20%|██        | 2/10 [00:18<01:17,  9.75s/it]

[2] pred=12 gold=12 ok=True | Step 1: Solve the system of equations for $n$ and $m$.
From the second equation, we have $n = 1 - m$.

Step 2: Substitute $n$ in the first equation.
$


MATH CoT:  30%|███       | 3/10 [00:33<01:24, 12.06s/it]

...

[2] pred=B gold=C ok=False | Step 1: Consider the properties of helium gas and how it behaves with temperature changes.
Step 2: Helium gas expands wh


ARC CoT:  30%|███       | 3/10 [00:15<00:35,  5.14s/it]

[3] pred=C gold=C ok=True | Step 1: Consider the impact of each option on the squirrel's habitat.
Step 2: New plant growth and changing of the seaso


ARC CoT: 100%|██████████| 10/10 [00:41<00:00,  4.18s/it]


✅ ARC CoT: 80.0% | Null=0

── Baseline Summary ─────────────────────────────────────
  MATH   zero_shot   : 40.0%
  MATH   cot         : 60.0%
  ARC    zero_shot   : 100.0%
  ARC    cot         : 80.0%


# CELL 17 — SLD-VM Evaluation: MATH

In [40]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 17 — SLD-VM Evaluation: MATH                        ║
# ╚══════════════════════════════════════════════════════════════╝
for k in STATS: STATS[k] = 0   # reset stats

correct, null_count = 0, 0
sldvm_math_records = []
print('Running SLD-VM on MATH...\n')

for i, s in enumerate(tqdm(math_samples, desc='SLD-VM MATH')):
    result = sldvm(s['problem'], 'math')
    pred = result.answer
    gold = extract_math_gold(s['solution'])
    ok   = math_correct(pred, gold)
    if ok:  correct += 1;  vmb.update_reliability(s['problem'], True)
    else:                  vmb.update_reliability(s['problem'], False)
    if pred is None: null_count += 1
    sldvm_math_records.append({'pred':pred,'gold':gold,'ok':ok,'s':s,'result':result})
    if i < 5:
        print(f'  [{i+1}] pred={pred} gold={gold} ok={ok} '
              f'bt={result.backtracks} llm={result.llm_calls} | {result.graph.summary()}')

acc = correct / len(math_samples)
RESULTS['MATH']['sldvm'] = acc
best_bl = max(RESULTS['MATH']['zero_shot'], RESULTS['MATH']['cot'])
delta = acc - best_bl

print(f'\n✅ SLD-VM MATH : {acc:.1%}')
print(f'   Best baseline: {best_bl:.1%}  Δ={delta:+.1%}  '
      f'{"✅ BEATS BASELINE" if delta > 0 else "❌ BELOW BASELINE"}')
print(f'   Backtracks   : {STATS["backtracks"]}')
print(f'   LLM calls    : {STATS["llm_calls"]}')
print(f'   Nodes pruned : {STATS["nodes_pruned"]}')
print(f'   {vmb.stats()}')

print("complete")

Running SLD-VM on MATH...



SLD-VM MATH:  10%|█         | 1/10 [00:15<02:20, 15.64s/it]

  [1] pred=\frac{1}{21} gold=\frac{18}{343} ok=False bt=0 llm=0 | Graph(7 nodes): ✅3 ❌0 ✂️4 ⏳0


SLD-VM MATH:  20%|██        | 2/10 [00:33<02:17, 17.22s/it]

  [2] pred=\frac{5}{2} gold=12 ok=False bt=0 llm=0 | Graph(7 nodes): ✅4 ❌0 ✂️3 ⏳0


...

  [4] pred=9 gold=\frac{7}{15} ok=False bt=0 llm=0 | Graph(7 nodes): ✅7 ❌0 ✂️0 ⏳0


SLD-VM MATH:  50%|█████     | 5/10 [01:28<01:25, 17.04s/it]

  [5] pred=7 gold=\dfrac{8}{7} ok=False bt=0 llm=0 | Graph(5 nodes): ✅5 ❌0 ✂️0 ⏳0


SLD-VM MATH: 100%|██████████| 10/10 [03:33<00:00, 21.33s/it]


✅ SLD-VM MATH : 0.0%
   Best baseline: 60.0%  Δ=-60.0%  ❌ BELOW BASELINE
   Backtracks   : 0
   LLM calls    : 1
   Nodes pruned : 21
   VMB: 11 patterns | avg_rel=0.77 | hits=5
complete


#  CELL 18 — SLD-VM Evaluation: ARC-Challenge

In [41]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 18 — SLD-VM Evaluation: ARC-Challenge               ║
# ╚══════════════════════════════════════════════════════════════╝
correct, null_count = 0, 0
sldvm_arc_records = []
print('Running SLD-VM on ARC-Challenge...\n')

for i, s in enumerate(tqdm(arc_samples, desc='SLD-VM ARC')):
    problem = format_arc(s)
    result  = sldvm(problem, 'arc', sample=s)
    pred = result.answer
    gold = arc_gold(s)
    ok   = arc_correct(pred, gold)
    if ok:  correct += 1;  vmb.update_reliability(problem, True)
    else:                  vmb.update_reliability(problem, False)
    if pred is None: null_count += 1
    sldvm_arc_records.append({'pred':pred,'gold':gold,'ok':ok,'s':s,'result':result})
    if i < 5:
        print(f'  [{i+1}] pred={pred} gold={gold} ok={ok} '
              f'bt={result.backtracks} llm={result.llm_calls} | {result.graph.summary()}')

acc = correct / len(arc_samples)
RESULTS['ARC']['sldvm'] = acc
best_bl = max(RESULTS['ARC']['zero_shot'], RESULTS['ARC']['cot'])
delta = acc - best_bl

print(f'\n✅ SLD-VM ARC  : {acc:.1%}')
print(f'   Best baseline: {best_bl:.1%}  Δ={delta:+.1%}  '
      f'{"✅ BEATS BASELINE" if delta > 0 else "❌ BELOW BASELINE"}')
print(f'   Backtracks   : {STATS["backtracks"]}')
print(f'   LLM calls    : {STATS["llm_calls"]}')
print(f'   {vmb.stats()}')

Running SLD-VM on ARC-Challenge...



SLD-VM ARC:  10%|█         | 1/10 [00:15<02:19, 15.54s/it]

  [1] pred=A gold=A ok=True bt=0 llm=0 | Graph(4 nodes): ✅4 ❌0 ✂️0 ⏳0


SLD-VM ARC:  20%|██        | 2/10 [00:25<01:36, 12.10s/it]

  [2] pred=C gold=C ok=True bt=0 llm=0 | Graph(4 nodes): ✅2 ❌0 ✂️2 ⏳0


...

  [4] pred=B gold=B ok=True bt=0 llm=0 | Graph(4 nodes): ✅2 ❌0 ✂️2 ⏳0


SLD-VM ARC:  50%|█████     | 5/10 [01:02<01:05, 13.18s/it]

  [5] pred=C gold=C ok=True bt=0 llm=0 | Graph(7 nodes): ✅4 ❌0 ✂️3 ⏳0


SLD-VM ARC: 100%|██████████| 10/10 [01:56<00:00, 11.68s/it]


✅ SLD-VM ARC  : 100.0%
   Best baseline: 100.0%  Δ=+0.0%  ❌ BELOW BASELINE
   Backtracks   : 0
   LLM calls    : 1
   VMB: 21 patterns | avg_rel=0.88 | hits=8


# CELL 19 — Final Results Table  

In [42]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 19 — Final Results Table                             ║
# ╚══════════════════════════════════════════════════════════════╝

print('=' * 70)
print(f'{"Method":<35} {"MATH":>10} {"ARC":>10}')
print('-' * 70)
for label, key in [('Baseline A — Zero-Shot','zero_shot'),
                   ('Baseline B — Chain-of-Thought','cot'),
                   ('SLD-VM (Full Pipeline)','sldvm')]:
    m = RESULTS['MATH'].get(key)
    a = RESULTS['ARC'].get(key)
    print(f'{label:<35} {(f"{m:.1%}" if m else "N/A"):>10} {(f"{a:.1%}" if a else "N/A"):>10}')
print('=' * 70)
print(f'{"Random baseline":35} {"N/A":>10} {"25.0%":>10}')
print('=' * 70)

print('\n── Delta vs Best Baseline ────────────────────────────────────────')
for ds in ['MATH','ARC']:
    bl  = max(RESULTS[ds]['zero_shot'], RESULTS[ds]['cot'])
    svm = RESULTS[ds].get('sldvm', 0)
    d   = svm - bl
    v   = '✅ BEATS BASELINE' if d > 0 else ('🟰 TIE' if d == 0 else '❌ BELOW')
    print(f'  {ds:6s}: best_bl={bl:.1%}  sldvm={svm:.1%}  Δ={d:+.1%}  {v}')

print('\n── Pipeline Efficiency ───────────────────────────────────────────')
print(f'  LLM (Qwen2.5-72B) calls: {STATS["llm_calls"]}')
print(f'  Total backtracks       : {STATS["backtracks"]}')
print(f'  Nodes pruned by VCE    : {STATS["nodes_pruned"]}')
print(f'  {vmb.stats()}')

with open('sldvm_results.json','w') as f:
    json.dump({'results':RESULTS,'stats':STATS,
               'model':CFG['small_model_id'],'verifier':CFG['verifier_model']}, f, indent=2)
print('\n✅ Results saved to sldvm_results.json')

Method                                    MATH        ARC
----------------------------------------------------------------------
Baseline A — Zero-Shot                   40.0%     100.0%
Baseline B — Chain-of-Thought            60.0%      80.0%
SLD-VM (Full Pipeline)                     N/A     100.0%
Random baseline                            N/A      25.0%

── Delta vs Best Baseline ────────────────────────────────────────
  MATH  : best_bl=60.0%  sldvm=0.0%  Δ=-60.0%  ❌ BELOW
  ARC   : best_bl=100.0%  sldvm=100.0%  Δ=+0.0%  🟰 TIE

── Pipeline Efficiency ───────────────────────────────────────────
  LLM (Qwen2.5-72B) calls: 1
  Total backtracks       : 0
  Nodes pruned by VCE    : 34
  VMB: 21 patterns | avg_rel=0.88 | hits=8

✅ Results saved to sldvm_results.json


In [43]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 20 — Failure Analysis                                ║
# ╚══════════════════════════════════════════════════════════════╝
print('═'*65)
print('FAILURE ANALYSIS — MATH (first 5 SLD-VM failures)')
print('═'*65)
for i, r in enumerate([r for r in sldvm_math_records if not r['ok']][:5]):
    s, res = r['s'], r['result']
    print(f'\n[{i+1}] {s.get("type","?")} | {s.get("level","?")}')
    print(f'  Problem: {s["problem"][:120]}')
    print(f'  pred={r["pred"]}  gold={r["gold"]}')
    print(f'  {res.graph.summary()}')
    for nid,node in list(res.graph.nodes.items())[:5]:
        icon = {'valid':'✅','invalid':'❌','pruned':'✂️','pending':'⏳'}[node.status.value]
        print(f'    {icon} [{node.node_type:10s}] {node.content[:70]}')
        if node.vce_reason: print(f'       VCE: {node.vce_reason[:80]}')

print()
print('═'*65)
print('FAILURE ANALYSIS — ARC (first 5 SLD-VM failures)')
print('═'*65)
for i, r in enumerate([r for r in sldvm_arc_records if not r['ok']][:5]):
    s, res = r['s'], r['result']
    print(f'\n[{i+1}] pred={r["pred"]}  gold={r["gold"]}')
    print(f'  Q: {s["question"][:120]}')
    print(f'  Choices: {s["choices"]["text"]}')
    print(f'  {res.graph.summary()}')
    for nid,node in list(res.graph.nodes.items())[:5]:
        icon = {'valid':'✅','invalid':'❌','pruned':'✂️','pending':'⏳'}[node.status.value]
        print(f'    {icon} [{node.node_type:10s}] {node.content[:70]}')

print()
print('═'*65)
print('MATH Error Type Breakdown')
print('═'*65)
type_stats = defaultdict(lambda: {'t':0,'c':0})
for r in sldvm_math_records:
    t = r['s'].get('type','Unknown')
    type_stats[t]['t'] += 1
    if r['ok']: type_stats[t]['c'] += 1
for t,st in sorted(type_stats.items()):
    print(f'  {t:30s}: {st["c"]}/{st["t"]} = {st["c"]/st["t"]:.1%}')

print()
h_ok = sum(1 for r in sldvm_arc_records if r['ok'] and r['result'].used_vmb)
h_n  = sum(1 for r in sldvm_arc_records if r['result'].used_vmb)
m_ok = sum(1 for r in sldvm_arc_records if r['ok'] and not r['result'].used_vmb)
m_n  = sum(1 for r in sldvm_arc_records if not r['result'].used_vmb)
print('VMB Contribution (ARC):')
if h_n: print(f'  With VMB hint : {h_ok}/{h_n} = {h_ok/h_n:.1%}')
if m_n: print(f'  Without VMB   : {m_ok}/{m_n} = {m_ok/m_n:.1%}')

═════════════════════════════════════════════════════════════════
FAILURE ANALYSIS — MATH (first 5 SLD-VM failures)
═════════════════════════════════════════════════════════════════

[1] Counting & Probability | Level 4
  Problem: Amy was born on a Tuesday. What is the probability that exactly two of her three best friends were also born on Tuesday?
  pred=\frac{1}{21}  gold=\frac{18}{343}
  Graph(7 nodes): ✅3 ❌0 ✂️4 ⏳0
    ✅ [premise   ] There are 7 days in a week.
       VCE: [PREMISE] accepted
    ✅ [deduction ] The probability of being born on a Tuesday is 1/7.
       VCE: [SM|conf=0.86] VERDICT: VALID
REASON: The probability of being born on a Tuesday
    ✅ [deduction ] The probability of not being born on a Tuesday is 6/7.
       VCE: [SM|conf=0.91] VERDICT: VALID
REASON: The probability of not being born on a Tue
    ✂️ [deduction ] We want exactly two friends to be born on Tuesday, so we need to calcu
       VCE: [SM|conf=0.83] VERDICT: INVALID
REASON: The prior step is incorre

# CELL 21 — Win/Loss Analysis: Where SLD-VM Helps 

In [44]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 21 — Win/Loss Analysis: Where SLD-VM Helps           ║
# ╚══════════════════════════════════════════════════════════════╝
for ds_name, sldvm_recs, cot_recs, samples in [
    ('MATH', sldvm_math_records, cot_math_records, math_samples),
    ('ARC',  sldvm_arc_records,  cot_arc_records,  arc_samples)
]:
    print('═'*65)
    print(f'SLD-VM vs CoT — {ds_name}')
    print('═'*65)
    cot_ok   = {i for i,r in enumerate(cot_recs) if r['ok']}
    sldvm_ok = {i for i,r in enumerate(sldvm_recs) if r['ok']}
    wins   = sldvm_ok - cot_ok
    losses = cot_ok   - sldvm_ok
    both   = sldvm_ok & cot_ok
    print(f'  SLD-VM wins (SLD-VM✅ CoT❌): {len(wins):3d}')
    print(f'  SLD-VM loses(SLD-VM❌ CoT✅): {len(losses):3d}')
    print(f'  Both correct               : {len(both):3d}')
    print(f'  Both wrong                 : {len(samples)-len(wins)-len(losses)-len(both):3d}')
    if wins:
        print(f'\nWin examples ({ds_name}):')
        for idx in list(wins)[:3]:
            r = sldvm_recs[idx]
            q = r["s"]["problem"][:90] if ds_name=="MATH" else r["s"]["question"][:90]
            print(f'  Q: {q}')
            print(f'  CoT={cot_recs[idx]["pred"]} SLD-VM={r["pred"]} Gold={r["gold"]}')
    print()

print('='*65)
print('FINAL VERDICT')
print('='*65)
for ds in ['MATH','ARC']:
    bl  = RESULTS[ds]['cot']
    svm = RESULTS[ds]['sldvm']
    d   = svm - bl
    v   = '✅ IMPROVEMENT' if d > 0.01 else ('🟰 SIMILAR' if abs(d) <= 0.01 else '❌ REGRESSION')
    print(f'  {ds:6s}: CoT={bl:.1%} → SLD-VM={svm:.1%}  Δ={d:+.1%}  {v}')

═════════════════════════════════════════════════════════════════
SLD-VM vs CoT — MATH
═════════════════════════════════════════════════════════════════
  SLD-VM wins (SLD-VM✅ CoT❌):   0
  SLD-VM loses(SLD-VM❌ CoT✅):   6
  Both correct               :   0
  Both wrong                 :   4

═════════════════════════════════════════════════════════════════
SLD-VM vs CoT — ARC
═════════════════════════════════════════════════════════════════
  SLD-VM wins (SLD-VM✅ CoT❌):   2
  SLD-VM loses(SLD-VM❌ CoT✅):   0
  Both correct               :   8
  Both wrong                 :   0

Win examples (ARC):
  Q: A party shop delivers helium-filled balloons to homes and businesses. The owners realize f
  CoT=B SLD-VM=C Gold=C
  Q: Which of the following is a characteristic of elements?
  CoT=B SLD-VM=D Gold=D

FINAL VERDICT
  MATH  : CoT=60.0% → SLD-VM=0.0%  Δ=-60.0%  ❌ REGRESSION
  ARC   : CoT=80.0% → SLD-VM=100.0%  Δ=+20.0%  ✅ IMPROVEMENT
